<a href="https://colab.research.google.com/github/janithcyapa/DHCA-Framework/blob/main/4.State_Estimator_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Validating State Estimation Using Extended Kalman Filter (EKF)

### Setup Environement

In [ ]:
!pip uninstall -y energy-plus-utility

In [ ]:
!pip install -q "energy-plus-utility @ git+https://github.com/janithcyapa/energy-plus-utility.git@main"
import importlib.metadata
ver = importlib.metadata.version("energy-plus-utility")
print(f"\n✅ Installed 'energy-plus-utility' version: {ver}")

In [ ]:
from eplus import prepare_colab_eplus
prepare_colab_eplus()

In [ ]:
!pip install control

## Setup Model

In [ ]:
from eplus.core import EPlusUtil
import types
import datetime

import pandas as pd
import numpy as np
import control as ct
import traceback


import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [63]:
# @title Setup Model
OUT_DIR = "/simulation/eplus_out"
url_idf="https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/5ZoneAirCooled.idf"
url_epw = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/Weather%20Files/LKA_Colombo-Katunayake.434500_SWERA.epw"

# Initialize Utility

sim = EPlusUtil(verbose=0, out_dir=OUT_DIR)
sim.reset_state()
sim.delete_out_dir()
sim.clear_eplus_outputs(patterns="eplusout.*")
sim.set_model_from_url(url_idf, url_epw)

sim.ensure_output_sqlite()
sim.prepare_run_with_co2(
    outdoor_co2_ppm=420.0,
    wipe_outputs=True,
    activate=True,
    reset=True
)
sim.run_dry_run(include_ems_edd=False,reset=True,design_day=True)

# catalog = sim.api_catalog_df()
# mo.ui.table(catalog)
# mo.ui.table(catalog['VARIABLES'])
# sim.list_available_variables()

0

## Setup Simulator

In [64]:
# @title Setup Data Logger
# Request the variables to construct State Vector (x_i) and Disturbances (d_i)
specs = [
    # Zone States (x_i)
    {"name": "Zone Mean Air Temperature", "key": "*"},       # T_in,i
    {"name": "Zone Mean Radiant Temperature", "key": "*"},   # T_m,i (Thermal Mass proxy)
    {"name": "Zone Mean Air Humidity Ratio", "key": "*"},    # W_in,i
    {"name": "Zone Air Relative Humidity", "key": "*"},      # W_in,i (%)
    {"name": "Zone Air CO2 Concentration", "key": "*"},      # ppm

    # Time-Varying Parameters (p_i)
    {"name": "Zone People Occupant Count", "key": "*"},      # No. of People
    {"name": "Zone Electric Equipment Total Heating Rate", "key": "*"}, # Watts

    # External Environment Conditions (x_out)
    {"name": "Site Outdoor Air Drybulb Temperature", "key": "*"}, # T_out
    {"name": "Site Outdoor Air Humidity Ratio", "key": "*"},      # W_out
    {"name": "Site Outdoor Air Relative Humidity", "key": "*"},   # W_in,i (%)
    {"name": "Schedule Value", "key": "CO2-Outdoor-Actuated"},     # ppm

    # Control Inputs - VAV Box Volumetric Flow Rate (m3/s)
    {"name": "System Node Current Density Volume Flow Rate", "key": "*"},

    # AHU Supply Parameters (S)
    {"name": "System Node Temperature", "key": "*"},       # T_s
    {"name": "System Node Humidity Ratio", "key": "*"},    # W_s
    {"name": "System Node CO2 Concentration", "key": "*"}, # C_s
]
sim.ensure_output_variables(specs, activate=True)

sim.collected_data = []
sim.current_state = {}

def state_logger(self, state):
    """Extracts sensor data, updates the current snapshot, and logs history."""
    if not self.exchange.api_data_fully_ready(state):
        return

    # 1. Get Simulation Time Details
    day = self.exchange.day_of_year(state)
    time_now = self.exchange.current_time(state)
    total_minutes = int(time_now * 60)
    hours, mins = divmod(total_minutes, 60)

    row = {
        "timestamp": f"Day {day:03d} {hours:02d}:{mins:02d}",
        "day": day,
        "hour": hours,
        "minute": mins,
        "time_decimal": time_now
    }

    # 2. Extract Outdoor and Supply Data
    t_out_h = self.exchange.get_variable_handle(state, "Site Outdoor Air Drybulb Temperature", "Environment")
    w_out_h = self.exchange.get_variable_handle(state, "Site Outdoor Air Humidity Ratio", "Environment")
    rh_out_h = self.exchange.get_variable_handle(state, "Site Outdoor Air Relative Humidity", "Environment")
    co2_out_h = self.exchange.get_variable_handle(state, "Schedule Value", "CO2-Outdoor-Actuated")

    row["T_out"] = self.exchange.get_variable_value(state, t_out_h)
    row["W_out"] = self.exchange.get_variable_value(state, w_out_h)
    row["RH_out_%"] = self.exchange.get_variable_value(state, rh_out_h)
    row["CO2_out"] = self.exchange.get_variable_value(state, co2_out_h)


    t_s_h = self.exchange.get_variable_handle(state, "System Node Temperature", "VAV Sys 1 Outlet Node")
    w_s_h = self.exchange.get_variable_handle(state, "System Node Humidity Ratio", "VAV Sys 1 Outlet Node")
    c_s_h = self.exchange.get_variable_handle(state, "System Node CO2 Concentration", "VAV Sys 1 Outlet Node")

    row["T_s"] = self.exchange.get_variable_value(state, t_s_h)
    row["W_s"] = self.exchange.get_variable_value(state, w_s_h)
    row["C_s"] = self.exchange.get_variable_value(state, c_s_h)

    # 3. Extract Internal States for All Zones
    zones = ["SPACE1-1", "SPACE2-1", "SPACE3-1", "SPACE4-1", "SPACE5-1"]
    for zone in zones:
        t_in_h = self.exchange.get_variable_handle(state, "Zone Mean Air Temperature", zone)
        t_m_h = self.exchange.get_variable_handle(state, "Zone Mean Radiant Temperature", zone)
        w_in_h = self.exchange.get_variable_handle(state, "Zone Mean Air Humidity Ratio", zone)
        rh_in_h = self.exchange.get_variable_handle(state, "Zone Air Relative Humidity", zone)
        co2_in_h = self.exchange.get_variable_handle(state, "Zone Air CO2 Concentration", zone)
        occ_in_h = self.exchange.get_variable_handle(state, "Zone People Occupant Count", zone)
        q_eq_h = self.exchange.get_variable_handle(state, "Zone Electric Equipment Total Heating Rate", zone)
        v_dot_h = self.exchange.get_variable_handle(state, "System Node Current Density Volume Flow Rate", f"{zone} In Node")

        row[f"{zone}_T_in"] = self.exchange.get_variable_value(state, t_in_h)
        row[f"{zone}_T_m"] = self.exchange.get_variable_value(state, t_m_h)
        row[f"{zone}_W_in"] = self.exchange.get_variable_value(state, w_in_h)
        row[f"{zone}_RH_%"] = self.exchange.get_variable_value(state, rh_in_h)
        row[f"{zone}_CO2_in"] = self.exchange.get_variable_value(state, co2_in_h)
        row[f"{zone}_Occ"] = self.exchange.get_variable_value(state, occ_in_h)
        row[f"{zone}_Q_equip"] = self.exchange.get_variable_value(state, q_eq_h)
        row[f"{zone}_V_dot"] = self.exchange.get_variable_value(state, v_dot_h)

    # 4. Update the current snapshot AND append to historical log
    self.current_state = row
    self.collected_data.append(row)

sim.state_logger = types.MethodType(state_logger, sim)
sim.register_handlers(
    "begin", [
        {"method_name": "state_logger"},
        {"method_name": "occupancy_handler", "kwargs": {"lam": 3.0, "min": 0, "max": 5, "seed": 4 } },
        {"method_name": "co2_set_outdoor_ppm", "kwargs": { "value_ppm": 420.0, "log_every_minutes": 60 } }
    ]
)

print(f"Handlers on 'begin' hook: {sim.list_handlers("begin")}")

Handlers on 'begin' hook: ['state_logger', 'occupancy_handler', 'co2_set_outdoor_ppm']


In [65]:
# @title zone_model
def zone_model(self, state):
    try:
        if not self.exchange.api_data_fully_ready(state):
            return
        zone_id = "SPACE1-1"

        # Time Stamping
        day = self.exchange.day_of_year(state)
        time = self.exchange.current_time(state)
        base_date = datetime.datetime(2026, 1, 1) + datetime.timedelta(days=day - 1, seconds=(int(time * 3600)))
        abs_time = (day * 24.0) + time

        # print("Zone :"+zone_id+" Time :"+str(base_date.strftime("%Y-%m-%d %H:%M:%S")))

        # Check If Zones is initialized
        if not hasattr(self, 'zones'):
            self.zones = {}

        # --- Initiallize Zone ---
        if zone_id not in self.zones:

            # --- Fetch Parameters and Initialize ---
            raw_params = self.get_zone_thermal_parameters()[zone_id]
            handles = {
                "T_in": self.exchange.get_variable_handle(state, "Zone Mean Air Temperature", zone_id),
                "T_m": self.exchange.get_variable_handle(state, "Zone Mean Radiant Temperature", zone_id),
                "W_in": self.exchange.get_variable_handle(state, "Zone Mean Air Humidity Ratio", zone_id),
                "CO2_in": self.exchange.get_variable_handle(state, "Zone Air CO2 Concentration", zone_id),
                "N_occ": self.exchange.get_variable_handle(state, "Zone People Occupant Count", zone_id),
                "T_out": self.exchange.get_variable_handle(state, "Site Outdoor Air Drybulb Temperature", "Environment"),
                "Q_equip": self.exchange.get_variable_handle(state, "Zone Electric Equipment Total Heating Rate", zone_id),
                "V_dot": self.exchange.get_variable_handle(state, "System Node Current Density Volume Flow Rate", f"{zone_id} In Node"),
                "T_s": self.exchange.get_variable_handle(state, "System Node Temperature", "VAV Sys 1 Outlet Node"),
                "W_s": self.exchange.get_variable_handle(state, "System Node Humidity Ratio", "VAV Sys 1 Outlet Node"),
                "C_s": self.exchange.get_variable_handle(state, "System Node CO2 Concentration", "VAV Sys 1 Outlet Node"),
            }

            inv_R_env_ext = 0.0
            R_env_gnd = None
            adj_zones = []

            for b in raw_params["boundaries"]:
              target = b["target"]
              r_abs = float(b["R_absolute_K_W"])
              if target == "Ground": R_env_gnd = r_abs
              elif target == "Environment" or b["boundary_condition"] == "outdoors": inv_R_env_ext += (1.0 / r_abs)
              else: adj_zones.append({ "zone": target, "R_env": r_abs, "handle_T_in": self.exchange.get_variable_handle( state, "Zone Mean Air Temperature", target )})
            R_env_ext = 1.0 / inv_R_env_ext if inv_R_env_ext > 0 else float('inf')


            # --- Define Dynamics ---
            def _dynamics(t, x, u, params):
                T_in, T_m, W_in, C_in = x # Extract State Variables
                V_dot_s = float(u[0]) # Extract Control Variable

                # consts
                rho_air, cp_air = 1.204, 1006.0
                q_person, g_w_person, g_co2_person = 100.0, 5e-5, 1e-5
                R_env_ext = float(params.get('R_env_ext', float('inf')))
                R_env_gnd = float(params.get('R_env_gnd', float('inf')))
                R_int = float(params['R_int'])
                C_air = float(params['C_air'])
                C_mass = float(params['C_mass'])
                M_air = float(params['M_air'])
                V_room = float(params['V_room'])


                T_s = params['T_s'] # Supply Variables
                W_s = params['W_s']
                C_s = params['C_s']

                N_occ = params['N_occ'] # Time-Varying Parameters
                Q_equip = params['Q_equip']

                T_out = params['T_out'] # External Temperature

                d_T, d_W, d_C = params['d_T'], params['d_W'], params['d_C'] # Disturbances


                # ----- Temperature Dynamics
                q_env = (T_out - T_in) / R_env_ext if R_env_ext < float('inf') else 0.0
                q_gnd = (22.0 - T_in) / R_env_gnd if R_env_gnd < float('inf') else 0.0
                q_adj = 0.0
                for adj in params['adj_zones']:
                    q_adj += (float(adj['T_in']) - float(T_in)) / float(adj['R_env'])
                q_mass = (T_m - T_in) / R_int
                q_int = (N_occ * q_person) + Q_equip
                q_s = rho_air * V_dot_s * cp_air * (T_s - T_in)
                dT_in_dt = (q_env + q_gnd + q_adj + q_mass + q_int + q_s + d_T) / C_air

                # ----- Mass Temperature Dynamics
                dT_m_dt = (T_in - T_m) / ( C_mass * R_int)

                # ----- Humidity Dynamics
                dot_m_s = rho_air * V_dot_s
                dW_in_dt = (N_occ * g_w_person + dot_m_s * (W_s - W_in) + d_W) / M_air

                # ----- CO2 Dynamics
                dC_in_dt = (N_occ * g_co2_person + V_dot_s * (C_s - C_in) + d_C) / V_room

                return np.array([dT_in_dt, dT_m_dt, dW_in_dt, dC_in_dt], dtype=float).flatten()

            def _outputs(t, x, u, params):
                # return [x[0], x[2], x[3]] # Observable: T_in, W_in, C_in
                return [x[0], x[1], x[2], x[3]]

            sys_ode = ct.NonlinearIOSystem(
                _dynamics, _outputs,
                inputs=['V_dot_s'],
                outputs=['T_in_obs','T_m_obs', 'W_in_obs', 'C_in_obs'],
                states=['T_in', 'T_m', 'W_in', 'C_in'],
                name=f'sys_{zone_id}'
            )

            # --- Store in Namespace ---
            self.zones[zone_id] = types.SimpleNamespace(
                last_time=abs_time,
                V_room=float(raw_params['V_room']),
                M_air=float(raw_params['M_air']),
                C_air=float(raw_params['C_air']),
                C_mass=float(raw_params['C_mass']),
                R_int=float(raw_params['R_int']),
                R_env_gnd=R_env_gnd,
                R_env_ext=R_env_ext,
                adj_zones=adj_zones,
                handles=handles,
                sys_ode=sys_ode,
                log=[]
            )

            # --- Testing Print ---
            z = self.zones[zone_id]
            print(f"\n--- [{zone_id}] Param Extraction ---")
            print(f"Physical: V={z.V_room}, M_air={z.M_air}")
            print(f"R_ground: {z.R_env_gnd}")
            print(f"R_env (External Merged): {z.R_env_ext:.6f}")
            print(f"Adjacent Zones Array: {z.adj_zones}")
            print("-----------------------------------\n")
            print(f"[{zone_id}] Handles initialized. Neighbors: {[az['zone'] for az in adj_zones]}")
            print(sys_ode)

        else:
            self.zones[zone_id].last_time = abs_time

        z = self.zones[zone_id]
        # --- Run Dynamic System
        row = {
            "timestamp": base_date.strftime("%Y-%m-%d %H:%M:%S"),
        }

        u_current = [ self.exchange.get_variable_value(state, z.handles["V_dot"]) ]

        current_adj_zones = []
        for adj in z.adj_zones:
            current_adj_zones.append({
                'T_in': self.exchange.get_variable_value(state, adj["handle_T_in"]),
                'R_env': adj["R_env"]
            })

        current_params = {
            'C_air': z.C_air, 'C_mass': z.C_mass,
            'R_env_ext': z.R_env_ext, 'R_env_gnd':z.R_env_gnd,
            'R_int': z.R_int, 'M_air': z.M_air, 'V_room': z.V_room,
            'T_out': self.exchange.get_variable_value(state, z.handles["T_out"]), # Dynamically updated disturbances from E+
            'N_occ': self.exchange.get_variable_value(state, z.handles["N_occ"]),
            'Q_equip': self.exchange.get_variable_value(state, z.handles["Q_equip"]),
            'T_s': self.exchange.get_variable_value(state, z.handles["T_s"]),# Supply Data
            'W_s': self.exchange.get_variable_value(state, z.handles["W_s"]),
            'C_s': self.exchange.get_variable_value(state, z.handles["C_s"]),
            'd_T': 0.0, 'd_W': 0.0, 'd_C': 0.0,# Observer variables (update these from your EKF later)
            'adj_zones': current_adj_zones # Adjacent Zones
            }

        has_pred = hasattr(z, 'prev_prediction')

        t_in_true = self.exchange.get_variable_value(state, z.handles["T_in"])
        t_m_true  = self.exchange.get_variable_value(state, z.handles["T_m"])
        w_in_true = self.exchange.get_variable_value(state, z.handles["W_in"])
        c_in_true = self.exchange.get_variable_value(state, z.handles["CO2_in"])
        x_actual = [t_in_true, t_m_true, w_in_true, c_in_true]
        has_pred = hasattr(z, 'prev_prediction')
        x_solver = [
            t_in_true,  # Measurable
            t_m_true,   # Measurable
            w_in_true,  # Measurable
            c_in_true   # Measurable
        ]

        dt_hours = self.exchange.system_time_step(state)
        if dt_hours == 0: # Fallback to zone step if system step isn't active yet
            dt_hours = self.exchange.zone_time_step(state)
        time_vector = [0, dt_hours * 3600.0]

        response = ct.input_output_response(z.sys_ode, time_vector, U=u_current, X0=x_solver, params=current_params)
        x_predicted_next = response.states[:, -1]

        row = {
            "timestamp": base_date,
            "T_in_actual": x_actual[0],
            "T_in_pred": z.prev_prediction[0] if has_pred else 0,
            "T_m_actual": x_actual[1],
            "T_m_pred": z.prev_prediction[1] if has_pred else 0,
            "W_in_actual": x_actual[2],
            "W_in_pred": z.prev_prediction[2] if has_pred else 0,
            "C_in_actual": x_actual[3],
            "C_in_pred": z.prev_prediction[3] if has_pred else 0,
            "T_out": current_params['T_out'],
            "V_dot_s": u_current[0]
        }
        z.log.append(row)
        z.prev_prediction = x_predicted_next

    except Exception as e:
        print(f"\n--- Python Exception in zone_1_model ---")
        print(f"Error: {e}")
        traceback.print_exc()
        print("----------------------------------------\n")

sim.zone_model = types.MethodType(zone_model, sim)
sim.register_handlers("begin", [{"method_name": "zone_model"}])

['state_logger', 'occupancy_handler', 'co2_set_outdoor_ppm', 'zone_model']

In [66]:
# @title zone_estimate_ekf
def zone_estimate_ekf(self, state):
    try:
        if not self.exchange.api_data_fully_ready(state):
            return
        zone_id = "SPACE1-1"

        # Time Stamping
        day = self.exchange.day_of_year(state)
        time = self.exchange.current_time(state)
        base_date = datetime.datetime(2026, 1, 1) + datetime.timedelta(days=day - 1, seconds=(int(time * 3600)))
        abs_time = (day * 24.0) + time

        # --- NOISE CONFIGURATION ---
        SIMULATE_NOISE = True
        SIMULATE_SUPPLY_NOISE = False

        # Standard Deviations (1-Sigma) derived from AHT21 & ENS160 Datasheets
        sigma_T = 0.1      # AHT21: +/- 0.3 C accuracy
        sigma_W = 0.0001   # AHT21: +/- 2% RH -> Approx 0.0003 kg/kg
        sigma_C = 25.0     # ENS160: eCO2 typically varies +/- 50 to 75 ppm

        # print("Zone_EKF :"+zone_id+" Time :"+str(base_date.strftime("%Y-%m-%d %H:%M:%S")))

        # Check If Zones is initialized
        if not hasattr(self, 'zones_ekf'):
            self.zones_ekf = {}

        # --- Initiallize Zone ---
        if zone_id not in self.zones_ekf:
          print("Initializing EKF for Zone : "+zone_id)

          # --- Fetch Parameters and Initialize ---
          raw_params = self.get_zone_thermal_parameters()[zone_id]
          handles = {
              "T_in": self.exchange.get_variable_handle(state, "Zone Mean Air Temperature", zone_id),
              "T_m": self.exchange.get_variable_handle(state, "Zone Mean Radiant Temperature", zone_id),
              "W_in": self.exchange.get_variable_handle(state, "Zone Mean Air Humidity Ratio", zone_id),
              "CO2_in": self.exchange.get_variable_handle(state, "Zone Air CO2 Concentration", zone_id),
              "N_occ": self.exchange.get_variable_handle(state, "Zone People Occupant Count", zone_id),
              "T_out": self.exchange.get_variable_handle(state, "Site Outdoor Air Drybulb Temperature", "Environment"),
              "Q_equip": self.exchange.get_variable_handle(state, "Zone Electric Equipment Total Heating Rate", zone_id),
              "V_dot": self.exchange.get_variable_handle(state, "System Node Current Density Volume Flow Rate", f"{zone_id} In Node"),
              "T_s": self.exchange.get_variable_handle(state, "System Node Temperature", "VAV Sys 1 Outlet Node"),
              "W_s": self.exchange.get_variable_handle(state, "System Node Humidity Ratio", "VAV Sys 1 Outlet Node"),
              "C_s": self.exchange.get_variable_handle(state, "System Node CO2 Concentration", "VAV Sys 1 Outlet Node"),
          }

          inv_R_env_ext = 0.0
          R_env_gnd = None
          adj_zones = []

          for b in raw_params["boundaries"]:
            target = b["target"]
            r_abs = float(b["R_absolute_K_W"])
            if target == "Ground": R_env_gnd = r_abs
            elif target == "Environment" or b["boundary_condition"] == "outdoors": inv_R_env_ext += (1.0 / r_abs)
            else: adj_zones.append({ "zone": target, "R_env": r_abs, "handle_T_in": self.exchange.get_variable_handle( state, "Zone Mean Air Temperature", target )})
          R_env_ext = 1.0 / inv_R_env_ext if inv_R_env_ext > 0 else float('inf')

          # EKF Matrices Initialization
          # State: [T_in, T_m, W_in, C_in, d_T, d_W, N_occ]
          P_est = np.eye(7) * 1.0  # Initial Covariance
          P_est[6,6] = 10

          # Q (Process Noise)
          Q = np.diag([0.1, 5.0, 1e-6, 10.0, 50.0, 1e-5, 1.0])
          # R (Measurement Noise) - Adapts based on noise toggle
          if SIMULATE_NOISE: R = np.diag([sigma_T**2, sigma_W**2, sigma_C**2]) # Variance = sigma^2
          else: R = np.diag([0.01, 1e-8, 1.0])

          # Observation Matrix H (We only measure T_in, W_in, C_in)
          H = np.zeros((3, 7))
          H[0, 0], H[1, 2], H[2, 3]  = 1.0, 1.0, 1.0

          self.zones_ekf[zone_id] = types.SimpleNamespace(
                last_time=abs_time,
                V_room=float(raw_params['V_room']),
                M_air=float(raw_params['M_air']),
                C_air=float(raw_params['C_air']),
                C_mass=float(raw_params['C_mass']),
                R_int=float(raw_params['R_int']),
                R_env_gnd=R_env_gnd,
                R_env_ext=R_env_ext,
                adj_zones=adj_zones,
                handles=handles,
                X_est=None,
                P_est=P_est,
                Q=Q,
                R=R,
                H=H,
                log=[]
            )


        z = self.zones_ekf[zone_id]

        # --- 1. Gather Current Measurements & Inputs ---
        t_in_meas = self.exchange.get_variable_value(state, z.handles["T_in"])
        w_in_meas = self.exchange.get_variable_value(state, z.handles["W_in"])
        c_in_meas = self.exchange.get_variable_value(state, z.handles["CO2_in"])

        # Extract Controls and Parameters
        V_dot_s = self.exchange.get_variable_value(state, z.handles["V_dot"])
        T_out = self.exchange.get_variable_value(state, z.handles["T_out"])
        Q_equip = self.exchange.get_variable_value(state, z.handles["Q_equip"])
        Q_equip = 0.0 # We don't know this in real life.
        T_s = self.exchange.get_variable_value(state, z.handles["T_s"])
        W_s = self.exchange.get_variable_value(state, z.handles["W_s"])
        C_s = self.exchange.get_variable_value(state, z.handles["C_s"])

        # Inject Noise into Sensored Data
        if SIMULATE_NOISE:
            t_in_meas += np.random.normal(0, sigma_T)
            w_in_meas += np.random.normal(0, sigma_W)
            c_in_meas += np.random.normal(0, sigma_C)
        if SIMULATE_SUPPLY_NOISE:
            T_s += np.random.normal(0, sigma_T)
            W_s += np.random.normal(0, sigma_W)
            C_s += np.random.normal(0, sigma_C)

        # Ground truths (For logging comparison ONLY)
        t_m_true  = self.exchange.get_variable_value(state, z.handles["T_m"])
        n_occ_true = self.exchange.get_variable_value(state, z.handles["N_occ"])

        # Constants
        rho_air, cp_air = 1.204, 1006.0
        q_person, g_w_person, g_co2_person = 100.0, 5e-5, 1e-5

        Z = np.array([t_in_meas, w_in_meas, c_in_meas]) # Measurements

        # First run: Initialize state with measurements and zero disturbances
        if z.X_est is None:
          # Init state: [T_in, T_m, W_in, C_in, d_T, d_W, N_occ]
          z.X_est = np.array([t_in_meas, t_in_meas, w_in_meas, c_in_meas, 0.0, 0.0, 0.0])
          z.last_time = abs_time
          return

        # Time step in seconds
        dt_hours = self.exchange.system_time_step(state)
        if dt_hours == 0:
          dt_hours = self.exchange.zone_time_step(state)
        dt = dt_hours * 3600.0
        if dt <= 0: return

        # Extract current state estimates
        T_in_e, T_m_e, W_in_e, C_in_e, d_T_e, d_W_e, N_occ_e = z.X_est


        # EKF ALGORITHUM
        # ___________________________________________________________________

        # --- 1.1 Prediction - Predict the State ---

        # ----- Temperature Dynamics Calcualtions
        q_env = (T_out - T_in_e) / z.R_env_ext if z.R_env_ext < float('inf') else 0.0
        q_gnd = (22.0 - T_in_e) / z.R_env_gnd if z.R_env_gnd < float('inf') else 0.0
        q_gnd = 0 # Remove Ground sinsce can't get measurements in practice
        _q_adj = 0.0
        inv_R_adj = 0.0
        for adj in z.adj_zones:
            t_adj = self.exchange.get_variable_value(state, adj["handle_T_in"])
            _q_adj += (t_adj) / float(adj["R_env"])
            inv_R_adj += 1.0 / float(adj["R_env"])
        q_adj = _q_adj - (T_in_e * inv_R_adj)
        q_mass = (T_m_e - T_in_e) / z.R_int
        q_int = (N_occ_e * q_person) + Q_equip
        q_s = rho_air * V_dot_s * cp_air * (T_s - T_in_e)

        # ----- Temperature Dynamics
        dT_in_dt = (q_env + q_gnd + q_adj + q_mass + q_int + q_s + d_T_e) / z.C_air
        # ----- Mass Temperature Dynamics
        dT_m_dt = (T_in_e - T_m_e) / (z.C_mass * z.R_int)
        # ----- Humidity Dynamics
        dW_in_dt = (N_occ_e * g_w_person + rho_air * V_dot_s * (W_s - W_in_e) + d_W_e) / z.M_air
        # ----- CO2 Dynamics
        dC_in_dt = (N_occ_e * g_co2_person + V_dot_s * (C_s - C_in_e)) / z.V_room

        # d_T, d_W, and N_occ are modeled as random walks (derivative = 0)
        X_pred = z.X_est + np.array([dT_in_dt, dT_m_dt, dW_in_dt, dC_in_dt, 0.0, 0.0, 0.0]) * dt


        # --- 1.2 Prediction - Predict the Covariance  ---

        # --- Jacobian Calculation (F matrix) ---
        df_dX = np.zeros((7, 7))

        # dT_in_dt partials
        inv_R_ext = 1.0 / z.R_env_ext if z.R_env_ext < float('inf') else 0.0
        inv_R_int = 1.0 / z.R_int if z.R_int < float('inf') else 0.0
        inv_R_gnd = 1.0 / z.R_env_gnd if z.R_env_gnd < float('inf') else 0.0
        inv_R_gnd = 0.0 # Remove Ground sinsce can't get measurements in practice

        df_dX[0, 0] = (-inv_R_ext - inv_R_gnd - inv_R_adj - inv_R_int - (rho_air * cp_air * V_dot_s)) / z.C_air
        df_dX[0, 1] = 1.0 / (z.C_air * z.R_int)
        df_dX[0, 4] = 1.0 / z.C_air
        df_dX[0, 6] = q_person / z.C_air

        # dT_m_dt partials
        df_dX[1, 0] = 1.0 / (z.C_mass * z.R_int)
        df_dX[1, 1] = -1.0 / (z.C_mass * z.R_int)

        # dW_in_dt partials
        df_dX[2, 2] = -(rho_air * V_dot_s) / z.M_air
        df_dX[2, 5] = 1.0 / z.M_air
        df_dX[2, 6] = g_w_person / z.M_air

        # dC_in_dt partials
        df_dX[3, 3] = -V_dot_s / z.V_room
        df_dX[3, 6] = g_co2_person / z.V_room

        F = np.eye(7) + df_dX * dt

        # Predict Covariance
        P_pred = F @ z.P_est @ F.T + z.Q

        # --- 2.1-3 Update - Calculate the Innovation, Innovation Covariance,Kalman Gain  ---
        y = Z - (z.H @ X_pred) # Innovation
        S = z.H @ P_pred @ z.H.T + z.R # Innovation Covariance
        K = P_pred @ z.H.T @ np.linalg.inv(S) # Kalman Gain

        # --- 2.4-5 Update - Update State and Covariance ---
        z.X_est = X_pred + K @ y
        z.P_est = (np.eye(7) - K @ z.H) @ P_pred

        # # Clip physical constraints (Cannot have negative people or negative absolute humidity)
        # z.X_est[2] = max(0.0, z.X_est[2]) # W_in
        # z.X_est[3] = max(0.0, z.X_est[3]) # C_in
        z.X_est[6] = max(0.0, z.X_est[6]) # N_occ

        # ___________________________________________________________________

        row = {
            "timestamp": base_date,
            "T_in_actual": t_in_meas,
            "T_in_pred": z.X_est[0],
            "T_m_actual": t_m_true,
            "T_m_pred": z.X_est[1],
            "W_in_actual": w_in_meas,
            "W_in_pred": z.X_est[2],
            "C_in_actual": c_in_meas,
            "C_in_pred": z.X_est[3],
            "N_occ_actual": n_occ_true,
            "N_occ_est": z.X_est[6],
            "d_T_est": z.X_est[4],
            "d_W_est": z.X_est[5],
            "T_out": T_out,
            "V_dot_s": V_dot_s
        }
        z.log.append(row)
        z.last_time = abs_time

    except Exception as e:
        print(f"\n--- Python Exception in zone_1_model ---")
        print(f"Error: {e}")
        traceback.print_exc()
        print("----------------------------------------\n")

sim.zone_estimate_ekf = types.MethodType(zone_estimate_ekf, sim)
sim.register_handlers("begin", [{"method_name": "zone_estimate_ekf"}])

['state_logger',
 'occupancy_handler',
 'co2_set_outdoor_ppm',
 'zone_model',
 'zone_estimate_ekf']

In [67]:
# @title Run Simulation
# Set Simulation Time
sim.set_simulation_params(
    start=(1, 1),
    end=(1, 7),
    timestep_per_hour = 12, # 4 (every 15 minutes) or 6 (every 10 minutes).
    start_day_of_week="Sunday",
)

print("Starting EnergyPlus Uncontrolled Simulation...")
res = sim.run_annual()

if(res == 0):
    print("Simulation Complete! Converting data to Pandas...")
    # Create the DataFrame
    SimulationData = pd.DataFrame(sim.collected_data)
    cols = ['timestamp'] + [c for c in SimulationData.columns if c != 'timestamp']
    SimulationData = SimulationData[cols]
    print("Done.")

if(res == 1):
    err_path = OUT_DIR / "eplusout.err"
    if err_path.exists():
        print("--- EnergyPlus Error Log ---")
        with open(err_path, 'r') as f:
            # Print the last 4000 characters to catch the fatal errors at the end
            print(f.read()[-4000:])
    else:
        print(f"Could not find the error file at: {err_path}")

Starting EnergyPlus Uncontrolled Simulation...

--- [SPACE1-1] Param Extraction ---
Physical: V=239.247360229, M_air=288.05
R_ground: 0.0023
R_env (External Merged): 0.004300
Adjacent Zones Array: [{'zone': 'PLENUM-1', 'R_env': 0.0081, 'handle_T_in': 23}, {'zone': 'SPACE2-1', 'R_env': 0.0199, 'handle_T_in': 27}, {'zone': 'SPACE4-1', 'R_env': 0.0199, 'handle_T_in': 31}, {'zone': 'SPACE5-1', 'R_env': 0.0045, 'handle_T_in': 33}]
-----------------------------------

[SPACE1-1] Handles initialized. Neighbors: ['PLENUM-1', 'SPACE2-1', 'SPACE4-1', 'SPACE5-1']
<NonlinearIOSystem>: sys_SPACE1-1
Inputs (1): ['V_dot_s']
Outputs (4): ['T_in_obs', 'T_m_obs', 'W_in_obs', 'C_in_obs']
States (4): ['T_in', 'T_m', 'W_in', 'C_in']

Update: <function zone_model.<locals>._dynamics at 0x7eb3f70fe7a0>
Output: <function zone_model.<locals>._outputs at 0x7eb3f70fccc0>
Initializing EKF for Zone : SPACE1-1
Simulation Complete! Converting data to Pandas...
Done.


## Results

In [53]:
# @title Helper Functions
def plot_zone_results(df):
    """
    Plots subplots comparing actual vs predicted state variables.
    Dynamically supports the base 4-state model and the 7-state EKF.
    Expects a DataFrame with 'timestamp' as the index.
    """

    # 1. Determine layout based on available columns
    has_n_occ = 'N_occ_est' in df.columns
    has_dist = 'd_T_est' in df.columns

    rows = 4
    titles = [
        "Zone Air Temperature (T_in)",
        "Thermal Mass Temperature (T_m)",
        "Humidity Ratio (W_in)",
        "CO2 Concentration (C_in)"
    ]
    specs = [[{"secondary_y": False}] for _ in range(4)]

    if has_n_occ:
        rows += 1
        titles.append("Occupancy Count (N_occ)")
        specs.append([{"secondary_y": False}])

    if has_dist:
        rows += 1
        titles.append("Estimated Disturbances (d_T and d_W)")
        specs.append([{"secondary_y": True}]) # Dual axis for different units

    # Create subplots
    fig = make_subplots(
        rows=rows, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        subplot_titles=titles,
        specs=specs
    )

    # Color palette
    colors = ['#1f77b4', '#d62728', '#2ca02c', '#ff7f0e', '#9467bd', '#e377c2', '#8c564b']

    # --- Standard States (Rows 1-4) ---
    fig.add_trace(go.Scatter(x=df.index, y=df['T_in_actual'], name='Actual T_in', line=dict(color=colors[0])), row=1, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['T_in_pred'], name='Predicted T_in', line=dict(color=colors[0], dash='dash')), row=1, col=1)

    fig.add_trace(go.Scatter(x=df.index, y=df['T_m_actual'], name='Actual T_m', line=dict(color=colors[1])), row=2, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['T_m_pred'], name='Predicted T_m', line=dict(color=colors[1], dash='dash')), row=2, col=1)

    fig.add_trace(go.Scatter(x=df.index, y=df['W_in_actual'], name='Actual W_in', line=dict(color=colors[2])), row=3, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['W_in_pred'], name='Predicted W_in', line=dict(color=colors[2], dash='dash')), row=3, col=1)

    fig.add_trace(go.Scatter(x=df.index, y=df['C_in_actual'], name='Actual C_in', line=dict(color=colors[3])), row=4, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['C_in_pred'], name='Predicted C_in', line=dict(color=colors[3], dash='dash')), row=4, col=1)

    current_row = 5

    # --- EKF States (Rows 5-6) ---
    if has_n_occ:
        fig.add_trace(go.Scatter(x=df.index, y=df['N_occ_actual'], name='Actual N_occ (E+)', line=dict(color=colors[4])), row=current_row, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df['N_occ_est'], name='Estimated N_occ', line=dict(color=colors[4], dash='dash')), row=current_row, col=1)
        fig.update_yaxes(title_text="People", row=current_row, col=1)
        current_row += 1

    if has_dist:
        # Primary Y: Thermal Disturbance (Watts)
        fig.add_trace(go.Scatter(x=df.index, y=df['d_T_est'], name='Est d_T (Thermal Dist)', line=dict(color=colors[5])), row=current_row, col=1, secondary_y=False)
        # Secondary Y: Moisture Disturbance (kg/s)
        fig.add_trace(go.Scatter(x=df.index, y=df['d_W_est'], name='Est d_W (Moisture Dist)', line=dict(color=colors[6], dash='dot')), row=current_row, col=1, secondary_y=True)

        fig.update_yaxes(title_text="d_T (W)", color=colors[5], row=current_row, col=1, secondary_y=False)
        fig.update_yaxes(title_text="d_W (kg/s)", color=colors[6], row=current_row, col=1, secondary_y=True)

    # Layout Updates
    fig.update_layout(
        height=250 * rows, # Dynamically scale height based on number of subplots
        title_text="RC Model vs EnergyPlus Validation",
        hovermode="x unified",
        template="plotly_dark",
        showlegend=True
    )

    # Standard Labels
    fig.update_yaxes(title_text="Temperature (°C)", row=1, col=1)
    fig.update_yaxes(title_text="Temperature (°C)", row=2, col=1)
    fig.update_yaxes(title_text="Humidity (kg/kg)", row=3, col=1)
    fig.update_yaxes(title_text="CO2 (ppm)", row=4, col=1)
    fig.update_xaxes(title_text="Time", row=rows, col=1) # Put X-axis label on the very last row

    fig.show()

# @title simulation_errors_summary
def simulation_errors_summary(df):
    """
    Takes the EnergyPlus vs RC Model DataFrame and calculates
    the numerical differences (errors) between actual and predicted states.
    Automatically detects EKF parameters.
    """
    print("=" * 65)
    print(f"{'MODEL VALIDATION ERROR SUMMARY':^65}")
    print("=" * 65)

    # 1. Define the pairs we want to analyze
    # Notice N_occ handles the '_est' naming convention from your logging dictionary
    variables = {
        'T_in':  {'actual': 'T_in_actual',  'pred': 'T_in_pred', 'unit': '°C',    'name': 'Zone Air Temp'},
        'T_m':   {'actual': 'T_m_actual',   'pred': 'T_m_pred',  'unit': '°C',    'name': 'Thermal Mass Temp'},
        'W_in':  {'actual': 'W_in_actual',  'pred': 'W_in_pred', 'unit': 'kg/kg', 'name': 'Humidity Ratio'},
        'C_in':  {'actual': 'C_in_actual',  'pred': 'C_in_pred', 'unit': 'ppm',   'name': 'CO2 Concentration'},
        'N_occ': {'actual': 'N_occ_actual', 'pred': 'N_occ_est', 'unit': 'count', 'name': 'Occupancy'}
    }

    results = []

    # 2. Calculate metrics for each variable
    for var_key, v in variables.items():
        # Check if the columns actually exist in the dataframe before calculating
        # This makes it safely ignore N_occ if an old model DataFrame is passed in
        if v['actual'] in df.columns and v['pred'] in df.columns:

            # Drop NaN values (like the very first timestep where pred is None)
            valid_data = df[[v['actual'], v['pred']]].dropna()

            if len(valid_data) == 0:
                continue

            # Calculate the raw error (Actual - Predicted)
            error = valid_data[v['actual']] - valid_data[v['pred']]

            # Calculate Statistics
            mae = error.abs().mean()
            max_err = error.abs().max()
            rmse = np.sqrt((error**2).mean())

            # Grab the error at the very last simulated timestep
            final_err = error.iloc[-1]

            results.append({
                'Variable': f"{v['name']} ({var_key})",
                'Unit': v['unit'],
                'MAE (Avg Error)': f"{mae:.4f}",
                'Max Error': f"{max_err:.4f}",
                'RMSE': f"{rmse:.4f}",
                'Final Step Error': f"{final_err:.4f}"
            })

    # 3. Format and print the results
    if not results:
        print("No prediction columns found to compare!")
        return None

    summary_df = pd.DataFrame(results)

    # Print the table nicely without the index
    print(summary_df.to_string(index=False))
    print("=" * 65)

    # Return the dataframe in case you want to save it to a CSV later
    return summary_df

In [60]:
# @title PART A: Process Base RC Model
zone_id = "SPACE1-1"

# ==========================================
# PART A: Process Base RC Model
# ==========================================
if hasattr(sim, 'zones') and zone_id in sim.zones:
    print(f"\nProcessing Base RC Model for {zone_id}...")

    # Extract and format DataFrame
    log_data_base = sim.zones[zone_id].log
    df_base_model = pd.DataFrame(log_data_base)
    df_base_model['timestamp'] = pd.to_datetime(df_base_model['timestamp'])
    df_base_model.set_index('timestamp', inplace=True)

    # Create a sliced DataFrame for summaries (Drop first hour)
    start_time_base = df_base_model.index.min()
    cutoff_time_base = start_time_base + pd.Timedelta(hours=1)
    df_base_summary = df_base_model[df_base_model.index >= cutoff_time_base]

    # Execute Visuals and Summaries
    plot_zone_results(df_base_model)  # Plot full timeline to see initialization
    simulation_errors_summary(df_base_summary)  # Summarize only post-convergence data
else:
    print(f"Base model data not found for {zone_id}.")


Processing Base RC Model for SPACE1-1...


                 MODEL VALIDATION ERROR SUMMARY                  
                Variable  Unit MAE (Avg Error) Max Error    RMSE Final Step Error
    Zone Air Temp (T_in)    °C          0.2308    1.5778  0.3465          -0.0383
 Thermal Mass Temp (T_m)    °C          0.0392    0.2664  0.0492          -0.0438
   Humidity Ratio (W_in) kg/kg          0.0001    0.0005  0.0001          -0.0001
CO2 Concentration (C_in)   ppm          9.7963   58.4082 11.3273          -4.1402


In [68]:
# @title PART B: Process EKF Model
zone_id = "SPACE1-1"

# ==========================================
# PART B: Process EKF Model
# ==========================================
if hasattr(sim, 'zones_ekf') and zone_id in sim.zones_ekf:
    print(f"\nProcessing EKF Model for {zone_id}...")

    # Extract and format DataFrame
    log_data_ekf = sim.zones_ekf[zone_id].log
    df_ekf_model = pd.DataFrame(log_data_ekf)
    df_ekf_model['timestamp'] = pd.to_datetime(df_ekf_model['timestamp'])
    df_ekf_model.set_index('timestamp', inplace=True)

    # Create a sliced DataFrame for summaries (Drop first hour)
    start_time_ekf = df_ekf_model.index.min()
    cutoff_time_ekf = start_time_ekf + pd.Timedelta(hours=1)
    df_ekf_summary = df_ekf_model[df_ekf_model.index >= cutoff_time_ekf]

    # Execute Visuals and Summaries
    plot_zone_results(df_ekf_model)  # Plot full timeline to see EKF convergence
    simulation_errors_summary(df_ekf_summary)  # Summarize only post-convergence data
else:
    print(f"EKF model data not found for {zone_id}.")


Processing EKF Model for SPACE1-1...


                 MODEL VALIDATION ERROR SUMMARY                  
                Variable  Unit MAE (Avg Error) Max Error    RMSE Final Step Error
    Zone Air Temp (T_in)    °C          0.0002    0.0019  0.0003          -0.0002
 Thermal Mass Temp (T_m)    °C          0.9431    2.7365  1.1867           0.1073
   Humidity Ratio (W_in) kg/kg          0.0000    0.0000  0.0000           0.0000
CO2 Concentration (C_in)   ppm         56.1125  167.8020 64.5685          20.3000
       Occupancy (N_occ) count          3.4993   12.2599  4.3761           2.0000
